In [1]:
from xgboost import XGBClassifier
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import roc_auc_score, f1_score, recall_score, precision_score, ConfusionMatrixDisplay, RocCurveDisplay
from sklearn.model_selection import StratifiedKFold, train_test_split
from imblearn.over_sampling import SMOTE, ADASYN, SMOTENC
import shap
import optuna


c:\Users\wenyu\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
df = pd.read_csv("data/train-cat-encoded.csv")
print("Training set size: ", df.shape)
y = df['Will_Buy_EV']
x = df.drop(columns=['Will_Buy_EV'])
x = x.drop(columns=['id', 'Home_Charging_Possible', 'City_Type_Rural', 'Current_Car_Type_Hatchback',
                    'Current_Car_Type_Truck', 'Gender_Male', 'Gender_Female', 'Gender_Other',
                    'Daily_Commute_km', 'Number_of_Cars_Owned', 'Charging_Stations_Near_Home',
                    'Charging_Stations_Near_Work', 'Range_Anxiety_Level'])
binary_features = [
    # 'Home_Charging_Possible', 
    'Subsidy_Available', 
    'City_Type_Urban', 'City_Type_Suburban', 
    # 'City_Type_Rural', 
    'Current_Car_Type_Sedan', 'Current_Car_Type_SUV', 
    # 'Current_Car_Type_Hatchback', 
    # 'Current_Car_Type_Truck', 
    # 'Gender_Male', 'Gender_Female', 'Gender_Other'
    ]
print(binary_features)


Training set size:  (668665, 22)
['Subsidy_Available', 'City_Type_Urban', 'City_Type_Suburban', 'Current_Car_Type_Sedan', 'Current_Car_Type_SUV']


In [3]:
n_splits = 5
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
subsetNum = 50000
subset_x = x.iloc[:subsetNum]
subset_y = y.iloc[:subsetNum]
smote = SMOTENC(
    random_state=42,
    categorical_features=binary_features)
ada = ADASYN(n_neighbors=5, random_state=42)


In [4]:
hyperparameters = {
    'n_estimators': [50, 100, 200, 300],
    'max_depth': [4, 6, 8, 10],
    'learning_rate': [0.02, 0.05, 0.1, 0.15],
    'subsample': [0.75, 0.8, 0.85],
    'colsample_bytree': [0.75, 0.8, 0.85]
}


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    subset_x, subset_y, test_size=0.2, random_state=42
)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)


In [6]:
print("==========XGBoost Cross Validation Training Loop==========")

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 300, step=50)
    max_depth = trial.suggest_categorical("max_depth", [4, 6, 8, 10])
    learning_rate = trial.suggest_categorical("learning_rate", [0.05, 0.1, 0.15])
    subsample = trial.suggest_categorical("subsample", [0.75, 0.8, 0.85])
    colsample_bytree = trial.suggest_categorical("colsample_bytree", [0.75, 0.8, 0.85])
    
    print(f"\n========== n_estimators {n_estimators} max_depth {max_depth}==========")
        
    # 1. Split data (using .iloc if X/y are pandas DataFrames)

    # X_train_resampled, y_train_resampled = ada.fit_resample(X_train, y_train)

    model = XGBClassifier(
    n_estimators=n_estimators,      # number of boosting rounds (trees)
    max_depth=max_depth,           # tree depth
    learning_rate=learning_rate,     # shrinkage per round
    subsample=subsample,         # row sampling per tree
    colsample_bytree=colsample_bytree,  # feature sampling per tree
    eval_metric='logloss'
)
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    auc = roc_auc_score(y_test, y_prob)
    f1 = f1_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)

    print(f"AUC: {auc:.4f}")
    print(f"F1: {f1:.4f} | Recall: {recall:.4f} | Precision: {precision:.4f}")

    return auc


==========XGBoost Cross Validation Training Loop==========


In [7]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)


[I 2026-09-11 15:11:35,836] A new study created in memory with name: no-name-069a9b58-6677-4087-a5ae-cb14af8a82b5



========== n_estimators 50 max_depth 4==========


[I 2026-09-11 15:11:36,238] Trial 0 finished with value: 0.9347044949506932 and parameters: {'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.85, 'colsample_bytree': 0.75}. Best is trial 0 with value: 0.9347044949506932.


AUC: 0.9347
F1: 0.6879 | Recall: 0.8238 | Precision: 0.5904

========== n_estimators 200 max_depth 10==========


[I 2026-09-11 15:11:37,763] Trial 1 finished with value: 0.9290901842694801 and parameters: {'n_estimators': 200, 'max_depth': 10, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.85}. Best is trial 0 with value: 0.9347044949506932.


...

AUC: 0.9361
F1: 0.6863 | Recall: 0.8109 | Precision: 0.5949

========== n_estimators 50 max_depth 4==========


[I 2026-09-11 15:12:34,700] Trial 98 finished with value: 0.9348667517349755 and parameters: {'n_estimators': 50, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.75, 'colsample_bytree': 0.8}. Best is trial 9 with value: 0.936588311749933.


AUC: 0.9349
F1: 0.6883 | Recall: 0.8244 | Precision: 0.5909

========== n_estimators 300 max_depth 10==========


[I 2026-09-11 15:12:36,551] Trial 99 finished with value: 0.9216520028333509 and parameters: {'n_estimators': 300, 'max_depth': 10, 'learning_rate': 0.15, 'subsample': 0.75, 'colsample_bytree': 0.8}. Best is trial 9 with value: 0.936588311749933.


AUC: 0.9217
F1: 0.6587 | Recall: 0.7096 | Precision: 0.6146


In [8]:
print(f"Best Score: {study.best_value}")
print(f"Best Parameters: {study.best_params}")
best = study.best_trial

print(f"Trial Number: {best.number}")
print(f"Best Score: {best.value}")
print(f"Best Params: {best.params}")


Best Score: 0.936588311749933
Best Parameters: {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.75}
Trial Number: 9
Best Score: 0.936588311749933
Best Params: {'n_estimators': 50, 'max_depth': 6, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 0.75}


### Without oversampling (10000 samples)
- Avg AUC: 0.7961
- Avg F1: 0.6768
- Avg Recall: 0.6476
- Avg Precision: 0.7091
- Std AUC: 0.011227993955145489

### With SMOTE 
- (50000 samples)
- Avg AUC: 0.9380
- Avg F1: 0.6928
- Avg Recall: 0.6803
- Avg Precision: 0.7061
- Std AUC: 0.0027307035663074416

- (100000 samples)
- Avg AUC: 0.9366
- Avg F1: 0.7012
- Avg Recall: 0.7416
- Avg Precision: 0.6650
- Std AUC: 0.001856941113581715

(200000 samples)
- Avg AUC: 0.9399
- Avg F1: 0.6997
- Avg Recall: 0.6903
- Avg Precision: 0.7095
- Std AUC: 0.0008880157870868589

(500000 samples)
- Avg AUC: 0.9402
- Avg F1: 0.6996
- Avg Recall: 0.6896
- Avg Precision: 0.7098
- Std AUC: 0.0008942851500033605

### With ADASYN 
- (50000 samples)
- Avg AUC: 0.9379
- Avg F1: 0.6930
- Avg Recall: 0.6819
- Avg Precision: 0.7048
- Std AUC: 0.0026846216264028214